# Brisk-Light Black Hole Imaging from GRMHD Simulations

Generates a movie using the brisk-light prescription (p = 0): one representative
GRMHD snapshot per lensing band, selected at the modal emission time t̄_n of
that band. Geodesics are computed once and reused for every frame.

In [1]:
using Jipole
using StaticArrays
using Printf

const MBH = 6.2e9

[ Info: Precompiling Jipole [2347fae0-cb17-4e37-b3ba-1e469be55413](cache misses: include_dependency fsize change (1))
[ Info: Precompiling Jipole [2347fae0-cb17-4e37-b3ba-1e469be55413] (cache misses: include_dependency fsize change (2))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up
[ Info: Chandra table loaded successfully from C:\Users\danyp\OneDrive\Escritorio\CosasPater\jipoleProyect\Jipole\src\models\..\..\Tables\ch24_vals.txt


6.2e9

In [2]:
#const all_dumps_path = "/home/pedro/kharma_dumps/tmp.%05d.h5"
const all_dumps_path = "../../data/kharma_dumps/tmp.%05d.h5"

"../../data/kharma_dumps/tmp.%05d.h5"

In [3]:
const dump_first = 0
const dump_last  = 10
const ImageCadence = 10

dump_list  = [Printf.format(Printf.Format(all_dumps_path), i) 
              for i in dump_first:dump_last]
dump_times = [Jipole.Brisklight.get_dump_time(i, all_dumps_path) for i in dump_first:dump_last]

params_brisklight = Jipole.Brisklight.OfBriskLight(3, zeros(Float64, 4), ImageCadence, 0.0)

println("Total dumps available: ", length(dump_list))
println("Time range: $(dump_times[1]) M  →  $(dump_times[end]) M")

Total dumps available: 11
Time range: 1100.0026804973966 M  →  2100.0055612048927 M


In [4]:
trat_large       = 20.
const trat_small     = 1.
const beta_crit      = 1.0
const th_beg         = 1.74e-2
const sigma_cut      = 1.0
const sigma_cut_high = -1.0

-1.0

In [5]:
const model = Jipole.Iharm.read_header(dump_list[1], MBH;
    th_beg=th_beg, trat_small=trat_small, beta_crit=beta_crit,
    sigma_cut=sigma_cut, sigma_cut_high=sigma_cut_high, 
    brisk_light=true, slow_light=false)

Initializing grid from: ../../data/kharma_dumps/tmp.00000.h5


custom electron model loaded from dump file...
Using Funky Modified Kerr-Schild coordinates FMKS
MKS parameters a: 0.937500 hslope: 0.300000 Rin: 1.001876 Rout: 1000.000000
FMKS parameters poly_xt: 0.820000 poly_alpha: 14.000000 mks_smooth: 0.500000 poly_norm: 0.757817
Grid start (startx): 1.874000951149755e-03, 0.000000000000000e+00, 0.000000000000000e+00 stop (stopx): 6.907755278982137e+00, 1.000000000000000e+00, 6.283185307179586e+00
grid dx: 5.395219748461709e-02, 7.812500000000000e-03, 6.283185307179586e+00


Jipole.Iharm.IharmParams(2, 2, 0, 1.666667, 1.3333333333333333, 1.6666666666666667, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 3.0e26, 30546.649412014707, 9.157655110892144e14, 6.2e9, 3.0, 3.906325282703709e-19, 351.08300776605216, 66.4216771242051, 0.9375, 0.3, 1.001875757988324, 1000.0, 0.82, 14.0, 0.5, 0.7578173169894967, 0.0, 0.0, 0.0, 0.0, 0.0, 128, 128, 1, [0.0, 0.05395219748461709, 0.0078125, 6.283185307179586], [0.0, 0.001874000951149755, 0.0, 0.0], [1.0, 6.907755278982137, 1.0, 6.283185307179586], [0.0, 0.0, 0.0, 0.0], [0.0, 6.907755278982137, 1.0, 6.283185307179586], 1.001875757988324, 100.0, 0.0174, 1.0, 1.0, 1.0, -1.0, false, true)

In [6]:
const ro      = 1000.0
const th      = 163.0
const phi     = 0.0

const res     = 128
const pixels_x = 128
const pixels_y = 128

const SourceD = 16.9e6 * Jipole.Constants.PC
const Rh      = 1 + sqrt(1. - model.a * model.a)
const freq    = 230e9

const DXsize  = SourceD / model.L_unit / Jipole.Constants.MUAS_PER_RAD * 160
const DYsize  = SourceD / model.L_unit / Jipole.Constants.MUAS_PER_RAD * 160
const fovx    = DXsize / ro
const fovy    = DYsize / ro

println("Resolution:        $pixels_x × $pixels_y pixels")
println("Inclination:       th = $(th)° (equivalent to $(90-th)° from jet axis)")
println("Spin:              a = $(model.a)")
println("Frequency:         230.0 GHz")
println("Field of view:     $(DXsize / model.L_unit / Jipole.Constants.MUAS_PER_RAD) μas")

Resolution:        128 × 128 pixels
Inclination:       th = 163.0° (equivalent to -73.0° from jet axis)
Spin:              a = 0.9375
Frequency:         230.0 GHz
Field of view:     2.338501443495858e-25 μas


In [7]:
using ProgressMeter

# Calculate the camera position in native coordinates
Xcamera = MVector{4,Float64}(Jipole.Camera.camera_position(ro, th, phi, model.a, model))

# Unitless frequency
const freq_unitless = freq * Jipole.Constants.HPL / (Jipole.Constants.ME * Jipole.Constants.CL * Jipole.Constants.CL)

# Array that will hold the band index for each pixel
midplane_crossings = zeros(Int, pixels_x, pixels_y)
nsteps = zeros(Int, pixels_x, pixels_y)

# Number of threads used in the calculation
println("Allocating workspaces for $pixels_x row-tasks...")
const maxnstep = 25000
dummy_svec = @SVector zeros(4)
dummy_traj = Jipole.GeoTypes.OfTrajS(0.0, dummy_svec, dummy_svec, dummy_svec, dummy_svec)

task_trajs = [Vector{Jipole.GeoTypes.OfTrajS}(undef, maxnstep) for _ in 1:pixels_x]
for i in 1:pixels_x
    for k in 1:maxnstep
        task_trajs[i][k] = dummy_traj
    end
end

# This will hold the exact number of steps for each pixel.
const all_geodesics = Matrix{Vector{Jipole.GeoTypes.OfTrajS}}(undef, pixels_x, pixels_y)

println("Tracing Geodesics...")
p = Progress(pixels_x * pixels_y; desc="Raytracing Image...", showspeed=true, barlen=30)

Threads.@threads for i in 0:(pixels_x - 1)
    my_traj = task_trajs[i + 1]
    
    for j in 0:(pixels_y - 1)
        nstep, midplane_crossings[i+1,j+1] = Jipole.Geodesics.get_pixel(
            my_traj, i, j, Xcamera, 
            fovx, fovy, freq_unitless, 
            pixels_x, pixels_y, model.a, 
            Rh, model.rmax_geo, model
        ) 
        nsteps[i+1, j+1] = nstep
        
        # Save to permanent storage
        all_geodesics[i + 1, j + 1] = my_traj[1:nstep]
        
        ProgressMeter.next!(p)
    end
end
finish!(p)

Allocating workspaces for 128 row-tasks...
Tracing Geodesics...


Raytracing Image... 100%|██████████████████████████████| Time: 0:00:23 ( 1.45 ms/it)it)


In [8]:
# Brisk-light parameters.
# image_cadence: time between output frames in units of M.
# t_obs_start / t_obs_end: observer-time range for the movie.

const t_obs_start = dump_times[1] + 50
const t_obs_end   = dump_times[end] - 50

# Initialize simulation_data as a vector of IharmData (one per band)
simulation_data = Vector{Jipole.Iharm.IharmData}(undef, params_brisklight.n_bands + 1)

println("Brisk-light configured:")
println("  Bands retained : $(params_brisklight.n_bands)")
println("  Image cadence  : $(params_brisklight.image_cadence) M")
println("  t_obs range    : $t_obs_start → $t_obs_end M")

Brisk-light configured:
  Bands retained : 3
  Image cadence  : 10.0 M
  t_obs range    : 1150.0026804973966 → 2050.0055612048927 M


In [9]:
#raw_lists = [Float64[] for _ in 0:params_brisklight.n_bands]
#for i in 1:pixels_x, j in 1:pixels_y
#    band_idx = midplane_crossings[i, j]
#    (band_idx < 0 || band_idx > params_brisklight.n_bands) && continue
#    n = nsteps[i, j]
#    n < 2 && continue
#    push!(raw_lists[band_idx + 1], all_geodesics[i, j][n].X[1])
#end
#for n in 0:params_brisklight.n_bands
#    ts = raw_lists[n + 1]
#    if isempty(ts)
#        println("banda $n: SIN PIXELES")
#    else
#        println("banda $n: n_pixeles=", length(ts),
#                "  rango X1 crudo = (", minimum(ts), ", ", maximum(ts), ")",
#                "  moda X1 cruda = ", Jipole.Brisklight.modal_hdi_kde(ts, 0.0).mode)
#    end
#end

In [ ]:
# Call the brisk-light renderer
Jipole.Brisklight.process_brisklight_images!(
    params_brisklight,
    simulation_data,
    all_geodesics,
    nsteps,
    midplane_crossings,
    model,
    pixels_x,
    pixels_y,
    freq,
    t_obs_start,
    t_obs_end,
    trat_large,
    all_dumps_path,
    dump_list,      
    dump_times      
)

[ Info: Brisk-light: band 0 → t̄_0 = -1096.714870832106 M  (144 pixels)
[ Info: Brisk-light: band 1 → t̄_1 = -1022.3041582095237 M  (15831 pixels)
[ Info: Brisk-light: band 2 → t̄_2 = -1043.227072470473 M  (390 pixels)


Loading data from '../../data/kharma_dumps/tmp.00000.h5' into 'Iharm' module...


[ Info: Brisk-light: band 3 → t̄_3 = -1058.9114957121012 M  (15 pixels)
[ Info: Brisk-light: processing frame at t_obs = 2246.7175513295024 M


All primitives successfully loaded. Dimensions: (128, 128, 1)
Loading data from '../../data/kharma_dumps/tmp.00001.h5' into 'Iharm' module...
All primitives successfully loaded. Dimensions: (128, 128, 1)
Loading data from '../../data/kharma_dumps/tmp.00002.h5' into 'Iharm' module...


[ Info:   Band window updated: t_target = 1150.0026804973963 M → dumps[(1, 2)] at t = [1100.0026804973966, 1200.0065090072605] M


...

[ Info: Brisk-light: processing frame at t_obs = 2786.7175513295024 M
Brisk-light t_obs = 2786.7175513295024 M 100%|██████████████████████████████| Time: 0:04:20 (15.88 ms/it)


Brisk-light: image saved → ../../data/Images/BriskLight\BriskImage.02787.txt


[ Info: Brisk-light: processing frame at t_obs = 2796.7175513295024 M
[ Info:   Band window updated: t_target = 1700.0026804973963 M → dumps[(7, 8)] at t = [1700.0013761363548, 1800.0015230265558] M
Brisk-light t_obs = 2796.7175513295024 M 100%|██████████████████████████████| Time: 0:03:54 (14.34 ms/it)
[ Info: Brisk-light: processing frame at t_obs = 2806.7175513295024 M


Brisk-light: image saved → ../../data/Images/BriskLight\BriskImage.02797.txt


Brisk-light t_obs = 2806.7175513295024 M   8%|███                           |  ETA: 0:04:22 (17.45 ms/it)

## Visualization

In [1]:
using DelimitedFiles
using CairoMakie
using Printf

brisk_dir  = "../../data/Images/BriskLight"
slow_dir   = "../../data/Images/SlowLight"
gif_out    = "C:\\Users\\danyp\\OneDrive\\Escritorio\\CosasPater\\jipoleProyect\\data\\Images\\comparison.gif"

brisk_files = sort(filter(f -> startswith(basename(f), "BriskImage.") && endswith(f, ".txt"),
                           readdir(brisk_dir, join=true)))
slow_files  = sort(filter(f -> startswith(basename(f), "Image.") && endswith(f, ".txt"),
                           readdir(slow_dir, join=true)))

println("Found $(length(brisk_files)) Brisk files and $(length(slow_files)) Slow files.")

# --- Extraer el índice numérico del nombre de archivo ---
function extract_index(path)
    m = match(r"(\d+)(?=\.txt$)", basename(path))
    return parse(Int, m.match)
end

brisk_indices = extract_index.(brisk_files)
slow_indices  = extract_index.(slow_files)

# --- Alinear el inicio de brisk al índice más cercano al primer índice de slow ---
slow_start = slow_indices[1]
closest_pos = argmin(abs.(brisk_indices .- slow_start))
println("Slow light empieza en índice $(slow_start). Brisk light se alinea desde índice $(brisk_indices[closest_pos]) (posición $(closest_pos) en la lista).")

brisk_files   = brisk_files[closest_pos:end]
brisk_indices = brisk_indices[closest_pos:end]

if length(brisk_files) != length(slow_files)
    println("The script will pair them up until it runs out of files in the shorter list.")
end

n_pairs = min(length(brisk_files), length(slow_files))
eps_val = 1e-30
vmin, vmax = 1e-12, 1e-5

fig = Figure(size=(1800, 500))

record(fig, gif_out, 1:n_pairs; framerate=10) do i
    empty!(fig)

    brisk_path = brisk_files[i]
    slow_path  = slow_files[i]
    brisk_filename = basename(brisk_path)
    slow_filename  = basename(slow_path)

    I_slow  = readdlm(slow_path)
    I_brisk = readdlm(brisk_path)
    if size(I_brisk) != size(I_slow)
        I_brisk = reshape(I_brisk, size(I_slow))
    end

    rel_err = abs.(I_brisk .- I_slow) ./ (abs.(I_slow) .+ eps_val)
    nmse = sum((I_slow .- I_brisk).^2) / sum(I_slow.^2)

    ax1 = Axis(fig[1, 1], title = "Slow Light | $(slow_filename)")
    hm1 = heatmap!(ax1, I_slow, colormap=:afmhot, colorscale=log10, colorrange=(vmin, vmax))
    Colorbar(fig[1, 2], hm1)

    ax2 = Axis(fig[1, 3], title = "Brisk Light | $(brisk_filename)")
    hm2 = heatmap!(ax2, I_brisk, colormap=:afmhot, colorscale=log10, colorrange=(vmin, vmax))
    Colorbar(fig[1, 4], hm2)

    ax3 = Axis(fig[1, 5], title = "Relative Error")
    hm3 = heatmap!(ax3, rel_err, colormap=:viridis, colorscale=log10, colorrange=(1e-6, 10.0))
    Colorbar(fig[1, 6], hm3)

    text!(ax3, 0.05, 0.95;
        text = @sprintf("NMSE = %.2e", nmse),
        space = :relative,
        color = :white,
        fontsize = 16,
        font = :bold,
        align = (:left, :top))

    println("[Pair $(lpad(i-1,4,'0'))] Frame added for $(brisk_filename) vs $(slow_filename)")
end

println("Done! GIF saved to $(gif_out)")

Found 56 Brisk files and 64 Slow files.
Slow light empieza en índice 2271. Brisk light se alinea desde índice 2267 (posición 3 en la lista).
The script will pair them up until it runs out of files in the shorter list.
[Pair 0000] Frame added for BriskImage.02267.txt vs Image.02271.txt
[Pair 0001] Frame added for BriskImage.02277.txt vs Image.02281.txt
[Pair 0002] Frame added for BriskImage.02287.txt vs Image.02291.txt
[Pair 0003] Frame added for BriskImage.02297.txt vs Image.02301.txt
[Pair 0004] Frame added for BriskImage.02307.txt vs Image.02311.txt
[Pair 0005] Frame added for BriskImage.02317.txt vs Image.02321.txt
[Pair 0006] Frame added for BriskImage.02327.txt vs Image.02331.txt
[Pair 0007] Frame added for BriskImage.02337.txt vs Image.02341.txt
[Pair 0008] Frame added for BriskImage.02347.txt vs Image.02351.txt
[Pair 0009] Frame added for BriskImage.02357.txt vs Image.02361.txt
[Pair 0010] Frame added for BriskImage.02367.txt vs Image.02371.txt
[Pair 0011] Frame added for BriskI